# Set Up

The code mounts Google Drive and sets the working directory to a specific folder for data and results storage. It then installs the SpeechBrain library (a toolkit for speech processing) and navigates to the speech enhancement template directory provided by SpeechBrain. Next, it loads the training hyperparameters from a YAML file (train.yaml). We specify to use GPU if available (or CPU if changed) and override some defaults like the number of training epochs. The hparams object will contain various settings such as file paths, model architecture, training parameters (e.g., sample rate, STFT settings, model definition, learning rate, etc.). This configuration drives the rest of the code.

In [ ]:
from google.colab import drive, output
import os

# drive mount
drive.mount("/content/")

In [ ]:
# Change working directory to a specific folder on Google Drive
%cd '/content/'

In [ ]:
%%capture

# Install SpeechBrain library (speech processing toolkit)
BRANCH = 'develop'
!python -m pip install git+https://github.com/speechbrain/speechbrain.git@$BRANCH

# (Optional) Could clone the SpeechBrain repository (using pip install above instead)
!git clone https://github.com/speechbrain/speechbrain/

In [ ]:
%cd speechbrain/templates/enhancement

## Load Hyperparameters

In [ ]:
import sys
import speechbrain as sb
from io import StringIO
from ruamel.yaml import YAML
from hyperpyyaml import load_hyperpyyaml

In [ ]:
hparams_file = './train.yaml'
run_opts = {'device': 'cuda:0'} # If you want to use CPU, set run_opts= {'device': 'cpu'}. But, we strongly recommend using a GPU.
overrides = {"number_of_epochs": 10,"num_workers" : 1}  # Change the desired value

In [ ]:
with open(hparams_file, encoding="utf-8") as fin:
  hparams = load_hyperpyyaml(fin, overrides)

You can check hyperparameters using this code.

In [ ]:
hparams

In [ ]:
# (Optional) Initialize DDP group for distributed training - has no effect for single GPU
sb.utils.distributed.ddp_init_group(run_opts)

## Create Experiment Directory

This section ensures the environment is set up for training. If using distributed training (multiple GPUs), ddp_init_group would handle initialization (in our single GPU or CPU case, it does nothing significant). Then, it creates an output directory (as specified in hparams["output_folder"]) to save model checkpoints, logs, and results. Ensuring this directory exists is important so that training can save files without error.

In [ ]:
experiment_directory = hparams["output_folder"]
if not os.path.isdir(experiment_directory):
    os.makedirs(experiment_directory)
print(f"Experiment directory created: {experiment_directory}")

## Prepare LibriSpeech

In [ ]:
import json
import os
import shutil

In [ ]:
data_folder = hparams["data_folder"]
save_json_train = hparams["train_annotation"]
save_json_valid = hparams["valid_annotation"]
save_json_test = hparams["test_annotation"]

In [ ]:
# create data path
train_folder = os.path.join(data_folder, "LibriSpeech", "train-clean-5") # ./data/LibriSpeech/train-clean-5
valid_folder = os.path.join(data_folder, "LibriSpeech", "dev-clean-2")   # ./data/LibriSpeech/ev-clean-2
test_folder = os.path.join(data_folder, "LibriSpeech", "test-clean")     # ./data/LibriSpeech/test-clean

In [ ]:
# dataset download
from speechbrain.utils.data_utils import download_file, get_all_files

train_archive = os.path.join(data_folder, "train-clean-5.tar.gz")
valid_archive = os.path.join(data_folder, "dev-clean-2.tar.gz")
test_archive = os.path.join(data_folder, "test-clean.tar.gz")

MINILIBRI_TRAIN_URL = "http://www.openslr.org/resources/31/train-clean-5.tar.gz"
MINILIBRI_VALID_URL = "http://www.openslr.org/resources/31/dev-clean-2.tar.gz"
MINILIBRI_TEST_URL = "https://www.openslr.org/resources/12/test-clean.tar.gz"

download_file(MINILIBRI_TRAIN_URL, train_archive)
download_file(MINILIBRI_VALID_URL, valid_archive)
download_file(MINILIBRI_TEST_URL, test_archive)

In [ ]:
# uppack tar.gz
shutil.unpack_archive(train_archive, data_folder)
shutil.unpack_archive(valid_archive, data_folder)
shutil.unpack_archive(test_archive, data_folder)

In [ ]:
# Create a list of files with the flac extension in the file list
extension = [".flac"]
wav_list_train = get_all_files(train_folder, match_and=extension)
wav_list_valid = get_all_files(valid_folder, match_and=extension)
wav_list_test = get_all_files(test_folder, match_and=extension)

We are not using the entire dataset; instead, we are using only 200 data points.

In [ ]:
wav_list_train = wav_list_train[:170]
wav_list_valid = wav_list_valid[:20]
wav_list_test = wav_list_test[:10]

Creates the json file given a list of wav files

In [ ]:
from speechbrain.dataio.dataio import read_audio

SAMPLERATE = 16000
def create_json(wav_list, json_file):
    """
    Arguments
    ---------
    wav_list : list of str
        The list of wav files.
    json_file : str
        The path of the output json file
    """
    # Processing all the wav files in the list
    json_dict = {}
    for wav_file in wav_list:

        # Reading the signal (to retrieve duration in seconds)
        signal = read_audio(wav_file)
        duration = signal.shape[0] / SAMPLERATE

        # Manipulate path to get relative path and uttid
        path_parts = wav_file.split(os.path.sep)
        uttid, _ = os.path.splitext(path_parts[-1])
        relative_path = os.path.join("{data_root}", *path_parts[-5:])

        # Create entry for this utterance
        json_dict[uttid] = {"wav": relative_path, "length": duration}

    # Writing the dictionary to the json file
    with open(json_file, mode="w", encoding="utf-8") as json_f:
        json.dump(json_dict, json_f, indent=2)
    print(f"{json_file} successfully created!")

In [ ]:
create_json(wav_list_train, save_json_train)
create_json(wav_list_valid, save_json_valid)
create_json(wav_list_test, save_json_test)

Create dataset objects "train" and "valid"

In [ ]:
# Load the signal, and pass it and its length to the corruption class

@sb.utils.data_pipeline.takes("wav")
@sb.utils.data_pipeline.provides("clean_sig")
def audio_pipeline(wav):
    clean_sig = sb.dataio.dataio.read_audio(wav)
    return clean_sig

In [ ]:
# Load train/valid/test datasets using JSON annotation files
datasets = {}
data_info = {
    "train": hparams["train_annotation"],
    "valid": hparams["valid_annotation"],
    "test": hparams["test_annotation"],
}

# DataLoader options: we set shuffle=False and later sort by length for efficient batching
hparams["dataloader_options"]["shuffle"] = False

for dataset in data_info:
    datasets[dataset] = sb.dataio.dataset.DynamicItemDataset.from_json(
        json_path=data_info[dataset],
        replacements={"data_root": hparams["data_folder"]},
        dynamic_items=[audio_pipeline],
        output_keys=["id", "clean_sig"],
    ).filtered_sorted(sort_key="length")

In [ ]:
datasets['train'][0]

## Download noise data

 We prepare the dataset for training, validation, and testing. The dataset of clean speech is described by JSON files (train.json, valid.json, test.json) which list audio file paths and metadata. We use SpeechBrain’s DynamicItemDataset to load these JSON files and attach an audio_pipeline that reads each audio file into a waveform tensor (clean_sig). Each dataset entry will contain an ID and the corresponding clean speech signal.

After loading the speech data, we download a noise dataset. Using prepare_dataset_from_URL, we fetch a zip of noise audio files and extract them into a directory (e.g., ./data/noise). This function also creates a CSV file (noise.csv) listing all noise files. These noise clips will be used to artificially add noise to clean speech during training (data augmentation), simulating noisy input for the enhancement model.

In [ ]:
from speechbrain.augment.preparation import prepare_dataset_from_URL

# Download and prepare noise data (download noise files to folder and create a list CSV)
URL =  'https://www.dropbox.com/scl/fi/a09pj97s5ifan81dqhi4n/noises.zip?rlkey=j8b0n9kdjdr32o1f06t0cw5b7&dl=1'
dest_folder = './data/noise'
ext = 'wav'
csv_file = 'noise.csv'

prepare_dataset_from_URL(URL, dest_folder, ext, csv_file)

# Modeling

## Define SEBrain Model

This is the core speech enhancement model definition. We create a class SEBrain that extends sb.Brain provided by SpeechBrain, which handles training and evaluation loops. The key methods overridden are:

**compute_forward**: Defines how to process a batch of data through the model to produce predictions.

    1.	It moves the batch to the correct device (CPU/GPU) and extracts clean_wavs (clean speech waveform) and lens (lengths).
    2.	It augments the clean speech with noise (wav_augment) to create noisy_wavs. This simulates a noisy input; essentially, we are mixing in a random noise from our noise dataset at a random SNR (signal-to-noise ratio) between the defined range (e.g., 0 to 15 dB) for training. This relates to data augmentation and creates the input for our enhancement model.
    3.	It computes spectral features from the noisy waveform. This involves taking the Short-Time Fourier Transform (STFT) to get a complex spectrogram, converting it to magnitude, and then using a log1p scaling (log(1 + magnitude)). Using log-spectral features compresses the dynamic range, which is helpful for the training stability and is conceptually related to methods like MMSE-LSA which operate in the log domain.
    4.	The processed noisy features are fed into the DNN model (self.modules.model which is our enhancement network, an LSTM-based model defined in the hyperparameters). The model outputs a mask (a tensor of the same shape as the spectral magnitude) which indicates for each frequency bin how much of the signal to keep. This is a DNN-based approach to speech enhancement: instead of using a fixed formula like the Wiener filter, the network learns an optimal mask to reduce noise. This mask is analogous to a Wiener filtering mask or a spectral gain function from classical enhancement methods.
    5.	We apply the predicted mask to the noisy spectral features to obtain a predicted clean spectrum (predict_spec). This technique is referred to as signal approximation (SA) because we directly try to make the masked noisy spectrum approximate the clean spectrum (as opposed to predicting the mask to approximate an ideal ratio mask, for instance). The result predict_spec is effectively the enhanced log-magnitude spectrum.
    6.	We then reconstruct the time-domain signal from this spectrum. We use the inverse STFT (ISTFT) via self.hparams.resynth. The resynth function uses the original noisy waveform’s phase information combined with the predicted magnitude (after converting the log magnitude back to linear scale with expm1). Using the noisy phase is a common and straightforward approach in spectral enhancement since our model does not predict the phase. The output predict_wav is the enhanced waveform (time-domain speech signal) that ideally has reduced noise. This waveform can later be listened to or saved to verify the enhancement effect.

**compute_feats**: A helper that computes the log-spectral features for a given waveform. It performs STFT, computes the magnitude, and applies torch.log1p. The comment notes that power=0.5 likely computes the magnitude (i.e., takes the square root of power spectrum if needed), and log1p reduces the emphasis on small differences (a small value in linear becomes much smaller in log scale relative to a large value).
  
**compute_objectives**: Defines the loss function for training by comparing predictions with targets.

    •	It first computes the target clean speech log-spectrum (clean_spec) from the clean waveform.
    •	The loss used is Mean Squared Error (MSE) between the predicted log-spectrum and the clean log-spectrum. Optimizing this objective means the model is learning to minimize the error in the log-magnitude domain, which closely aligns with the goal of MMSE-LSA (Minimum Mean Square Error Log-Spectral Amplitude) estimators in classical speech enhancement. In other words, the network is trying to produce a spectral amplitude estimate that is as close as possible to the clean speech amplitude on a log scale.
    •	We record the loss for each batch for monitoring (using SpeechBrain’s MetricStats).
    •	Additionally, for validation and test stages, we compute the STOI (Short-Time Objective Intelligibility) metric between the enhanced speech (predictions["wav"]) and the clean speech (self.clean_wavs). STOI is a common metric to evaluate speech intelligibility after enhancement (higher STOI means the speech is more intelligible).
    •	The method returns the loss, which will be used for backpropagation.

**on_stage_start** and **on_stage_end**: These are lifecycle methods called at the beginning and end of each epoch (or evaluation run).

    •	On stage start, we initialize metrics (like resetting the loss tracker, and preparing STOI metric tracker for validation/test).
    •	On stage end, if training, we store the epoch’s average loss. If validating or testing, we compute the average stats (loss and STOI). For validation:
    •	It logs the stats for the epoch (both training loss and validation loss/STOI).
    •	It uses the checkpointer to save the model checkpoint, keeping only the best ones (here it will keep the checkpoint with the highest STOI, since we provide max_keys=["stoi"] and note we use negative STOI in stats so “highest STOI” corresponds to lowest negative STOI).
    •	For test stage, it will log the final test loss and STOI as well.

In [ ]:
import torch

# Define SEBrain class: Inherits SpeechBrain's Brain to manage the training/evaluation loop
class SEBrain(sb.Brain):
    """Class that manages the training loop. See speechbrain.core.Brain."""

    def compute_forward(self, batch, stage):
        """Apply masking to convert from noisy waveforms to enhanced signals.

        Arguments
        ---------
        batch : PaddedBatch
            This batch object contains all the relevant tensors for computation.
        stage : sb.Stage
            One of sb.Stage.TRAIN, sb.Stage.VALID, or sb.Stage.TEST.

        Returns
        -------
        predictions : dict
            A dictionary with keys {"spec", "wav"} with predicted features.
        """
        # We first move the batch to the appropriate device, and
        # compute the features necessary for masking.
        batch = batch.to(self.device)
        self.clean_wavs, self.lens = batch.clean_sig

        noisy_wavs, self.lens = self.hparams.wav_augment(
            self.clean_wavs, self.lens
        )

        noisy_feats = self.compute_feats(noisy_wavs)

        # Masking is done here with the "signal approximation (SA)" algorithm.
        # The masked input is compared directly with clean speech targets.
        mask = self.modules.model(noisy_feats)
        predict_spec = torch.mul(mask, noisy_feats)

        # Also return predicted wav, for evaluation. Note that this could
        # also be used for a time-domain loss term.
        predict_wav = self.hparams.resynth(
            torch.expm1(predict_spec), noisy_wavs
        )

        # Return a dictionary so we don't have to remember the order
        return {"spec": predict_spec, "wav": predict_wav}

    def compute_feats(self, wavs):
        """Returns corresponding log-spectral features of the input waveforms.

        Arguments
        ---------
        wavs : torch.Tensor
            The batch of waveforms to convert to log-spectral features.

        Returns
        -------
        feats : torch.Tensor
            The computed features.
        """
        # Log-spectral features
        feats = self.hparams.compute_STFT(wavs)
        feats = sb.processing.features.spectral_magnitude(feats, power=0.5)

        # Log1p reduces the emphasis on small differences
        feats = torch.log1p(feats)

        return feats

    def compute_objectives(self, predictions, batch, stage):
        """Computes the loss given the predicted and targeted outputs.

        Arguments
        ---------
        predictions : dict
            The output dict from `compute_forward`.
        batch : PaddedBatch
            This batch object contains all the relevant tensors for computation.
        stage : sb.Stage
            One of sb.Stage.TRAIN, sb.Stage.VALID, or sb.Stage.TEST.

        Returns
        -------
        loss : torch.Tensor
            A one-element tensor used for backpropagating the gradient.
        """
        # Prepare clean targets for comparison
        clean_spec = self.compute_feats(self.clean_wavs)

        # Directly compare the masked spectrograms with the clean targets
        loss = sb.nnet.losses.mse_loss(
            predictions["spec"], clean_spec, self.lens
        )

        # Append this batch of losses to the loss metric for easy
        self.loss_metric.append(
            batch.id,
            predictions["spec"],
            clean_spec,
            self.lens,
            reduction="batch",
        )

        # Some evaluations are slower, and we only want to perform them
        # on the validation set.
        if stage != sb.Stage.TRAIN:

            # Evaluate speech intelligibility as an additional metric
            self.stoi_metric.append(
                batch.id,
                predictions["wav"],
                self.clean_wavs,
                self.lens,
                reduction="batch",
            )

        return loss

    def on_stage_start(self, stage, epoch=None):
        """Gets called at the beginning of each epoch.

        Arguments
        ---------
        stage : sb.Stage
            One of sb.Stage.TRAIN, sb.Stage.VALID, or sb.Stage.TEST.
        epoch : int
            The currently-starting epoch. This is passed
            `None` during the test stage.
        """
        # Set up statistics trackers for this stage
        self.loss_metric = sb.utils.metric_stats.MetricStats(
            metric=sb.nnet.losses.mse_loss
        )

        # Set up evaluation-only statistics trackers
        if stage != sb.Stage.TRAIN:
            self.stoi_metric = sb.utils.metric_stats.MetricStats(
                metric=sb.nnet.loss.stoi_loss.stoi_loss
            )

    def on_stage_end(self, stage, stage_loss, epoch=None):
        """Gets called at the end of an epoch.

        Arguments
        ---------
        stage : sb.Stage
            One of sb.Stage.TRAIN, sb.Stage.VALID, sb.Stage.TEST
        stage_loss : float
            The average loss for all of the data processed in this stage.
        epoch : int
            The currently-starting epoch. This is passed
            `None` during the test stage.
        """
        # Store the train loss until the validation stage.
        if stage == sb.Stage.TRAIN:
            self.train_loss = stage_loss

        # Summarize the statistics from the stage for record-keeping.
        else:
            stats = {
                "loss": stage_loss,
                "stoi": -self.stoi_metric.summarize("average"),
            }

        # At the end of validation, we can write stats and checkpoints
        if stage == sb.Stage.VALID:
            # The train_logger writes a summary to stdout and to the logfile.
            self.hparams.train_logger.log_stats(
                {"Epoch": epoch},
                train_stats={"loss": self.train_loss},
                valid_stats=stats,
            )

            # Save the current checkpoint and delete previous checkpoints,
            # unless they have the current best STOI score.
            self.checkpointer.save_and_keep_only(meta=stats, max_keys=["stoi"])

        # We also write statistics about test data to stdout and to the logfile.
        if stage == sb.Stage.TEST:
            self.hparams.train_logger.log_stats(
                {"Epoch loaded": self.hparams.epoch_counter.current},
                test_stats=stats,
            )

## Initialization

Initialize the Brain object to prepare for mask training.

We instantiate SEBrain with the components defined in our hyperparameters:

	•	modules includes our CustomModel (the DNN for masking, composed of LSTM and Linear layers with ReLU as defined in the YAML).
	•	opt_class is the optimizer (here it’s an Adam optimizer configured with a learning rate, also from YAML).
	•	We pass hparams for it to access all other hyperparameters (like STFT functions, augmentation, etc.), and run_opts to ensure it knows to use the GPU.
	•	We also pass the checkpointer which is set up to save model checkpoints in the specified folder.



In [ ]:
se_brain = SEBrain(
    modules=hparams["modules"],
    opt_class=hparams["opt_class"],
    hparams=hparams,
    run_opts=run_opts,                        # 'cpu' or 'cuda:0'
    checkpointer=hparams["checkpointer"],     # ./results/<seed>/save
)

## Training

The `fit()` method iterates the training loop, calling the methods
necessary to update the parameters of the model. Since all objects
with changing state are managed by the Checkpointer, training can be
stopped at any point, and will be resumed on next call.

In [ ]:
# @title
se_brain.fit(
    epoch_counter=se_brain.hparams.epoch_counter,
    train_set=datasets["train"],
    valid_set=datasets["valid"],
    train_loader_kwargs=hparams["dataloader_options"],
    valid_loader_kwargs=hparams["dataloader_options"],
)

# Evaluation

Load best checkpoint (highest STOI) for evaluation

In [ ]:
test_stats = se_brain.evaluate(
    test_set=datasets["test"],
    max_key="stoi",
    test_loader_kwargs=hparams["dataloader_options"],
)

# Inference

At this point, we can also use the model for inference on any noisy input. For example, given a new noisy speech waveform, we could feed it through se_brain (or use se_brain.hparams.compute_STFT, apply the model mask, and resynth) to get an enhanced speech waveform. The predict_wav obtained in compute_forward is the enhanced audio. In an interactive environment (like a notebook), we could play this audio to hear the difference before and after enhancement. The approach implemented (spectral mask estimation via a DNN) is a modern deep-learning-based speech enhancement technique, connecting the theory from the PPT (Wiener filter concept, MMSE estimator in log domain, etc.) with practical implementation.

In [ ]:
import librosa
import soundfile as sf
from IPython.display import Audio
import numpy as np

In [ ]:
noisy_audio_path = '/content/speechbrain/templates/enhancement/data/noise/pointsource_noises/noise-free-sound-0105.wav'
wav1,sr1 = librosa.load(noisy_audio_path)
print(f'waveform : {wav1}')
print(f'sample rate : {sr1}\n')
Audio(noisy_audio_path)

In [ ]:
clean_audio_path = '/content/speechbrain/templates/enhancement/data/LibriSpeech/train-clean-5/1088/134318/1088-134318-0010.flac'
wav2,sr2 = librosa.load(clean_audio_path)
print(f'waveform : {wav2}')
print(f'sample rate : {sr2}\n')
Audio(clean_audio_path)

In [ ]:
if sr1 != sr2:
  raise ValueError("Sample rates do not match")

max_len = max(len(wav1), len(wav2))
wav1_padded = np.pad(wav1, (0, max_len - len(wav1)))
wav2_padded = np.pad(wav2, (0, max_len - len(wav2)))

combined_audio = 0.5 * wav1_padded + wav2_padded
combined_audio = combined_audio / np.max(np.abs(combined_audio))
Audio(combined_audio,rate = sr1)

In [ ]:
sf.write("/content/speechbrain/templates/enhancement/data/noisy_audio_sample.wav", combined_audio, sr1)

## Load Pretrained Model

Use pre-trained models for Inference.

In [ ]:
from speechbrain.pretrained import SpectralMaskEnhancement

enhance_model = SpectralMaskEnhancement.from_hparams(
    source="speechbrain/metricgan-plus-voicebank",
    savedir="pretrained_models/metricgan-plus-voicebank",
    run_opts={"device": "cuda"}
)

In [ ]:
enhance_model

## Result

This audio is the audio after speech enhancement.

In [ ]:
noisy_audio_path = "/content/speechbrain/templates/enhancement/data/noisy_audio_sample.wav"
waveform, sr = torchaudio.load(noisy_audio_path)
waveform = waveform.to("cuda")

In [ ]:
enhanced = enhance_model.enhance_batch(waveform, lengths=torch.tensor([1.0], device="cuda"))
enhanced = enhanced.cpu().squeeze().numpy()
Audio(enhanced, rate=sr)